# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SammyFarrelZebua/flyrank-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Plain-Words Contract Answers for CTR / Engagement Opportunity Scoring Lane:

1. **Unit of Analysis (Grain):** One row = one content item (`content_id`) for a specific client (`client_id`) evaluated over a trailing 90-day observation window.
2. **Tables Used:** `data/raw/content_refresh_anonymized.csv` (starter dataset, 30,000 rows × 44 columns across 32 anonymized clients).
3. **Time Window:** Trailing 90-day aggregate window per row. Sub-windows (`impressions_last_30d` vs `impressions_prev_30d`) measure 30-day momentum.
4. **Target / Label:** `ctr` — observed click-through rate from Google Search Console (`clicks ÷ impressions`), stored as a ×100 percentage (e.g., `0.76` = 0.76%). The derived target opportunity score is `(expected_ctr_for_position_tier - actual_ctr) * impressions_90d / 100.0`.
5. **Deliberately Excluded:** 
   - `clicks_90d`, `clicks_last_30d`, `clicks_prev_30d`: Direct target leakage (clicks is the numerator of CTR; using it to predict CTR is circular).
   - `trend_direction`, `trend_pct`, `is_declining_label`: Label source for the content refresh lane, excluded here to prevent cross-lane target leakage.
   - `content_id`, `client_id`: Pseudonymized context identifiers, strictly reserved for grouping and client-level train-test splits (never used as model inputs).

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# ── Section 1 Verification: grain and shape ──────────────────────────────────
print("=== Section 1: Unit of Analysis Verification ===\n")
print(f"Total rows:          {len(df):,}")
print(f"Unique content_id:   {df['content_id'].nunique():,}")
print(f"Unique client_id:    {df['client_id'].nunique():,}")
print(f"Columns:             {df.shape[1]}")
print(f"\nCTR summary (×100 percentage — 0.76 means 0.76%, NOT 76%):")
print(df['ctr'].describe().to_string())

=== Section 1: Unit of Analysis Verification ===

Total rows:          30,000
Unique content_id:   30,000
Unique client_id:    32
Columns:             44

CTR summary (×100 percentage — 0.76 means 0.76%, NOT 76%):
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification — Four Buckets

| Bucket | Fields | Available-when? / Justification |
|---|---|---|
| **Feature** (5 max) | `avg_position` | Knowable at decision moment because GSC records daily search position before evaluation. Note: `avg_position == 0` means NO data, not rank zero (1,205 rows). |
| **Feature** | `impressions_90d` | Knowable at decision moment because it is a trailing 90-day aggregate logged before prediction point. |
| **Feature** | `word_count` | Knowable at decision moment because content length is a static, published property of the URL (blank for ~7,699 rows). |
| **Feature** | `content_age_days` | Knowable at decision moment because publication timestamp predates the evaluation window. |
| **Feature** | `search_volume` | Knowable at decision moment because target keyword search volume is historical demand from search index data (blank for ~2,468 rows with no keyword data). |
| **Label / Target** | `ctr` | Observed GSC click-through rate — the outcome we predict/rank by. Stored as ×100 (e.g., `0.76` = 0.76%). |
| **Context** | `content_id`, `client_id` | Pseudonymized IDs for grouping, joining, and client-holdout splitting. Never used as model inputs. |
| **Context** | `content_type`, `position_tier` | Categorical context used for tier-mean baseline and stratification. |
| **Excluded** | `clicks_90d`, `clicks_last_30d`, `clicks_prev_30d` | **Direct target leakage**: `ctr = clicks / impressions × 100`. Clicks is the literal numerator of the label — using it as a feature makes the prediction circular and trivially perfect. |
| **Excluded** | `trend_direction`, `trend_pct`, `is_declining_label` | **Cross-lane leakage**: these are the label source for the Content Refresh lane. Using them here imports a different lane's target signal into our features. |
| **Excluded** | `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | **Multi-system denominator mismatch**: these GA4 rates can exceed 100% (different measurement systems); not safe as direct features without normalisation. Can be revisited in later modeling weeks. |

In [2]:
# Building feature frame
features = ['avg_position', 'impressions_90d', 'word_count', 'content_age_days', 'search_volume']
label = 'ctr'
context = ['content_id', 'client_id', 'content_type', 'position_tier']

# Extract just the relevant slice
df_work = df[features + [label] + context].copy()

print("Feature frame shape:", df_work.shape)
print("\nFeature frame dtypes:")
print(df_work[features].dtypes)
print("\nFirst 3 rows:")
display(df_work[features + [label]].head(3))

Feature frame shape: (30000, 10)

Feature frame dtypes:
avg_position        float64
impressions_90d       int64
word_count          float64
content_age_days      int64
search_volume       float64
dtype: object

First 3 rows:


,avg_position,impressions_90d,word_count,content_age_days,search_volume,ctr
0,10.6,3803,3221.0,187,10.0,0.76
1,20.3,15320,2481.0,445,90.0,0.05
2,36.5,12581,3515.0,141,0.0,0.09


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Contract Facts & Query Verification

We verify three core claims on the data:
1. **Grain Uniqueness:** Confirm that `content_id` is unique across the entire dataset (i.e. grain is indeed 1 row = 1 content item).
2. **Row Counts and Client Coverage:** Verify total row count, client coverage, and distribution of observations.
3. **Availability (avg_position > 0 IS TRUE):** Verify how many rows survive once we filter out invalid search position data (`avg_position == 0` represents "no data", not rank zero).
4. **Systematic Missingness Investigation:** Check the missingness rate by `content_type` to show that it is systematic rather than random (especially for `word_count` and `search_volume`).

In [5]:
# ── QUERY 1: Grain Uniqueness ────────────────────────────────────────────────
dupes = df.groupby('content_id').size()
n_dupes = (dupes > 1).sum()
print(f"Q1 — Grain check: duplicate content_id rows = {n_dupes}")
assert n_dupes == 0, "Grain uniqueness violated!"
print("     PASS: one row per content_id confirmed.\n")

# ── QUERY 2: Row count & date span / age span ────────────────────────────────
total_rows = len(df)
valid_imp_rows = (df['impressions_90d'] > 0).sum()
n_clients = df['client_id'].nunique()
age_min, age_max = df['content_age_days'].min(), df['content_age_days'].max()
print(f"Q2 — Row count: {total_rows:,} total rows | {valid_imp_rows:,} with impressions_90d > 0")
print(f"     Clients: {n_clients} | content_age_days range: {age_min:.0f} – {age_max:.0f} days\n")

# ── QUERY 3: Availability (avg_position > 0 IS TRUE) ──────────────────────────
# Per data-dictionary: avg_position == 0 means NO search position data (not rank zero)
available_rows = (df['avg_position'] > 0).sum()
zero_pos_rows = (df['avg_position'] == 0).sum()
print(f"Q3 — Availability (avg_position > 0 IS TRUE): {available_rows:,} rows survive")
print(f"     Rows excluded (avg_position == 0, no GSC rank data): {zero_pos_rows:,}")
print(f"     Availability rate: {available_rows / total_rows:.1%}\n")

# ── QUERY 4: Systematic Missingness Check ─────────────────────────────────────
print("Q4 — Missingness rates across feature set:")
missing_rates = df[features].isnull().mean()
for col, rate in missing_rates.items():
    print(f"     {col:20s}: {rate:6.1%} missing ({df[col].isnull().sum():,}/{total_rows:,} rows)")

print("\nMissingness of word_count grouped by content_type:")
print(df.groupby('content_type')['word_count'].apply(lambda x: x.isnull().mean()).to_string())

Q1 — Grain check: duplicate content_id rows = 0
     PASS: one row per content_id confirmed.

Q2 — Row count: 30,000 total rows | 30,000 with impressions_90d > 0
     Clients: 32 | content_age_days range: 90 – 564 days

Q3 — Availability (avg_position > 0 IS TRUE): 28,795 rows survive
     Rows excluded (avg_position == 0, no GSC rank data): 1,205
     Availability rate: 96.0%

Q4 — Missingness rates across feature set:
     avg_position        :   0.0% missing (0/30,000 rows)
     impressions_90d     :   0.0% missing (0/30,000 rows)
     word_count          :  25.7% missing (7,699/30,000 rows)
     content_age_days    :   0.0% missing (0/30,000 rows)
     search_volume       :   8.2% missing (2,468/30,000 rows)

Missingness of word_count grouped by content_type:
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.282979


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Primary Data Limitation

**CTR varies non-linearly by position tier — observed data cannot reveal the underlying root cause.**

1. **Root Cause Ambiguity:** Observed search data records *what* happened (low CTR at position 3), but cannot reveal *why* (mismatched page title, weak meta description, intent mismatch, strong brand competition, or SERP feature displacement). The model functions strictly as a decision-support candidate generator, not a causal diagnostic.
2. **Missing Position Data (1,205 rows):** ~4.0% of rows have `avg_position == 0` (indicating no GSC search rank data during the observation window). These pages cannot be evaluated against position-aware CTR benchmarks.
3. **Systematic Missingness:** `word_count` is missing for 25.7% of rows (7,699 pages) and `search_volume` is missing for 8.2% of rows (2,468 pages). This missingness is non-random (driven by `content_type`); simple `fillna(0)` would silently inject categorical information into numeric features.

---

### The Leakage Experiment ("The Trap")

Below, we demonstrate the leakage lesson:
1. Train an honest baseline model to predict `ctr` using the 5 clean features. Record honest MAE.
2. Inject a leaky feature (`clicks_90d` — the literal numerator of CTR = `clicks ÷ impressions × 100`).
3. Observe the artificial score jump toward zero MAE.
4. Remove `clicks_90d`, confirm the honest baseline is restored, and document why the leaky feature was invalid.

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Prepare data for modeling experiment (dropping nulls in honest features for clean test)
honest_features = ['avg_position', 'impressions_90d', 'word_count', 'content_age_days', 'search_volume']
df_exp = df[honest_features + ['ctr', 'clicks_90d']].dropna()

X_honest = df_exp[honest_features]
y = df_exp['ctr']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_honest, y, test_size=0.2, random_state=42
)

# ── 1. HONEST MODEL ──────────────────────────────────────────────────────────
rf_honest = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
rf_honest.fit(X_train_h, y_train_h)
preds_honest = rf_honest.predict(X_test_h)
mae_honest = mean_absolute_error(y_test_h, preds_honest)
print(f"1. HONEST MODEL MAE:   {mae_honest:.4f}%")

# ── 2. INJECT LEAKY FEATURE (clicks_90d = CTR numerator) ─────────────────────
# ctr = (clicks_90d / impressions_90d) * 100
X_leaky = df_exp[honest_features + ['clicks_90d']]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

rf_leaky = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
rf_leaky.fit(X_train_l, y_train_l)
preds_leaky = rf_leaky.predict(X_test_l)
mae_leaky = mean_absolute_error(y_test_l, preds_leaky)
print(f"2. LEAKED MODEL MAE:   {mae_leaky:.4f}%  ← Artificial near-perfect jump!")

print(f"\n   Score Jump: MAE dropped from {mae_honest:.4f}% down to {mae_leaky:.4f}%")
print("   WARNING: This jump is fake! Including clicks_90d leaks the target numerator.")

# ── 3. PURGE LEAKY FEATURE ───────────────────────────────────────────────────
del rf_leaky, X_leaky
print(f"\n3. PURGED & VERIFIED: Leaky feature removed. Honest MAE retained: {mae_honest:.4f}%")

1. HONEST MODEL MAE:   0.3822%
2. LEAKED MODEL MAE:   0.0456%  ← Artificial near-perfect jump!

   Score Jump: MAE dropped from 0.3822% down to 0.0456%

3. PURGED & VERIFIED: Leaky feature removed. Honest MAE retained: 0.3822%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.